# EcoVision — treino do detector e avaliação na TACO (Google Colab)

Este notebook executa, na ordem:

1. baixa a base **TACO** e seleciona as classes **lata, garrafa e sacola**;
2. divide as imagens em **treino / validação / teste** (sem imagem repetida entre divisões);
3. converte para o formato YOLO e **treina o YOLO11** na GPU;
4. roda a **comparação original × CLAHE × gama** no conjunto de teste, com o mesmo detector,
   e gera métricas (Precision, Recall, F1, mAP@0.5), resultados por classe, por iluminação e imagens de exemplo.

Tudo que importa (pesos, métricas, imagens) é salvo no **Google Drive**, em `MyDrive/EcoVision`.

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.

## 0. Configuração

In [ ]:
# Ajuste aqui se necessário
DRIVE_DIR   = "/content/drive/MyDrive/EcoVision"
CODE_ZIP    = f"{DRIVE_DIR}/ProjetoPDI_EcoVision.zip"   # opcional: o zip pode ser enviado direto ao Colab (painel Arquivos)
REPO_URL    = ""          # alternativa: URL do GitHub (ex.: https://github.com/usuario/ProjetoPDI_EcoVision.git)
REPO_BRANCH = "feature/backend-taco"

MODELO_BASE = "yolo11s.pt"   # yolo11n.pt = mais rápido; yolo11s.pt = melhor equilíbrio na T4
IMGSZ       = 1024           # resíduos costumam ser pequenos na foto; 640 perde detalhe
EPOCHS      = 100
PATIENCE    = 25             # para cedo se a validação não melhorar
BATCH       = 16             # reduza para 8 se faltar memória de GPU
SEED        = 42
GAMMA_TESTE = 1.25           # gama usada na condição "gamma" da comparação
CONF        = 0.25           # limiar de confiança para Precision/Recall/F1
RUN_NAME    = f"{MODELO_BASE.split('.')[0]}_taco3_{IMGSZ}"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 1. Código do projeto e dependências

In [ ]:
import os, shutil, subprocess
PROJ = "/content/EcoVision"
if os.path.exists(PROJ):
    shutil.rmtree(PROJ)

if REPO_URL:
    subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, PROJ], check=True)
else:
    # Procura o zip: 1) enviado pelo painel "Arquivos" do Colab (/content),
    # 2) no Drive; 3) se não achar, abre o botão de upload.
    import glob
    candidatos = sorted(glob.glob("/content/*EcoVision*.zip")) + ([CODE_ZIP] if os.path.exists(CODE_ZIP) else [])
    if candidatos:
        zip_projeto = candidatos[0]
    else:
        from google.colab import files
        print("Selecione o zip do projeto (ProjetoPDI_EcoVision.zip):")
        enviado = files.upload()
        zip_projeto = "/content/" + next(iter(enviado))
    print("Usando", zip_projeto)
    if os.path.exists("/content/_code"):
        shutil.rmtree("/content/_code")
    shutil.unpack_archive(zip_projeto, "/content/_code")
    raiz = next(os.path.join(dp) for dp, dn, fn in os.walk("/content/_code") if "requirements.txt" in fn)
    shutil.move(raiz, PROJ)

%cd {PROJ}
!pip install -q ultralytics pydantic-settings fastapi python-multipart
!ls

## 2. Base TACO

Na primeira execução baixa as ~1.500 fotos (10–20 min) e guarda a parte usada no Drive.
Nas próximas execuções restaura do Drive e pula o download.

In [ ]:
import os, shutil
CACHE = f"{DRIVE_DIR}/cache_dataset.zip"

if os.path.exists(CACHE):
    print("Restaurando dataset do Drive...")
    shutil.unpack_archive(CACHE, f"{PROJ}/dataset")
    BAIXAR = False
else:
    BAIXAR = True
    !git clone -q https://github.com/pedropro/TACO.git /content/TACO
    %cd /content/TACO
    !pip install -q requests pillow
    !python download.py --dataset_path ./data/annotations.json
    %cd {PROJ}
print("BAIXAR =", BAIXAR)

## 3. Selecionar classes e dividir treino / validação / teste

Mapeamento (em `scripts/preparar_base_taco.py`, dicionário `CATEGORIAS_TACO`):

| classe | categorias TACO |
|---|---|
| lata | Drink can, Food Can |
| garrafa | Clear plastic bottle, Glass bottle, Other plastic bottle |
| sacola | Garbage bag, Single-use carrier bag, Polypropylene bag |

Na TACO isso dá cerca de 263 latas, 439 garrafas e 95 sacolas anotadas. Sacola é a classe com menos exemplos:
espere métricas menos estáveis para ela e cite isso nas limitações.

Divisão: 70% treino, 10% validação (usada só para escolher a melhor época), 20% teste (só para as métricas finais).

In [ ]:
if BAIXAR:
    !python -m scripts.preparar_base_taco \
        --anotacoes /content/TACO/data/annotations.json \
        --imagens /content/TACO/data \
        --saida dataset/subset_3classes \
        --treino 0.7 --validacao 0.1 --seed {SEED} --copiar --sobrescrever
    !python -m scripts.converter_yolo --base dataset/subset_3classes --saida dataset/yolo_3classes --max-lado 1600
    print("Salvando cache no Drive...")
    shutil.make_archive(CACHE[:-4], "zip", f"{PROJ}/dataset")
else:
    # o data.yaml guarda caminho absoluto; regrava para este ambiente
    from pathlib import Path
    yaml_path = Path("dataset/yolo_3classes/data.yaml")
    linhas = yaml_path.read_text().splitlines()
    linhas[0] = f"path: {yaml_path.parent.resolve().as_posix()}"
    yaml_path.write_text("\n".join(linhas) + "\n")

!cat dataset/yolo_3classes/data.yaml
for split in ("train", "val", "test"):
    n = len(os.listdir(f"dataset/yolo_3classes/images/{split}"))
    print(split, n, "imagens")

## 4. Treino

Treina **somente com imagens originais** (sem CLAHE/gama). Assim o mesmo modelo é usado nas três condições
e a diferença medida depois vem apenas do pré-processamento.

Observação para a apresentação: o Ultralytics aplica por padrão aumento de dados de cor/brilho (`hsv_v=0.4`),
o que deixa o modelo mais robusto a variações de iluminação e pode reduzir o ganho do pré-processamento.
Isso é uma limitação a discutir, não um erro.

Se o Colab desconectar, rode a célula de novo: ela retoma do último checkpoint salvo no Drive.

In [ ]:
import os
from ultralytics import YOLO

RUNS = f"{DRIVE_DIR}/runs"
last = f"{RUNS}/{RUN_NAME}/weights/last.pt"
concluido = f"{RUNS}/{RUN_NAME}/weights/treino_concluido.txt"

if os.path.exists(concluido):
    print("Treino já concluído; pulando. Apague", concluido, "para treinar de novo.")
elif os.path.exists(last):
    print("Retomando treino de", last)
    YOLO(last).train(resume=True)
else:
    YOLO(MODELO_BASE).train(
        data=f"{PROJ}/dataset/yolo_3classes/data.yaml",
        imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        seed=SEED, deterministic=True,
        project=RUNS, name=RUN_NAME, exist_ok=True,
        plots=True,
    )
open(concluido, "w").close()
BEST = f"{RUNS}/{RUN_NAME}/weights/best.pt"
print("Pesos finais:", BEST)

In [ ]:
# Curvas de treino geradas pelo Ultralytics (úteis para os slides)
from IPython.display import Image, display
for nome in ("results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"):
    caminho = f"{RUNS}/{RUN_NAME}/{nome}"
    if os.path.exists(caminho):
        print(nome); display(Image(caminho, width=900))

## 5. Conferência rápida no teste (Ultralytics, imagens originais)

Só para checar o treino. Os números oficiais do trabalho vêm da etapa 6.

In [ ]:
BEST = f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt"
m = YOLO(BEST).val(data=f"{PROJ}/dataset/yolo_3classes/data.yaml", split="test",
                   imgsz=IMGSZ, project=f"{DRIVE_DIR}/runs", name=f"{RUN_NAME}_test", exist_ok=True)
print(f"mAP@0.5 (teste, original): {m.box.map50:.3f}")

## 6. Comparação original × CLAHE × gama (resultado do trabalho)

Mesmo detector, mesmos pesos, mesmo limiar, IoU ≥ 0,50 — exatamente o protocolo da Etapa 1.
Gera, em `MyDrive/EcoVision/resultados`:

- `taco_metrics.csv` — P, R, F1 e AP@0.5 por condição e por classe;
- `detalhes/por_iluminacao.csv` — as mesmas métricas em imagens escuras, intermediárias e claras;
- `detalhes/por_imagem.csv` — acertos e erros de cada imagem;
- `detalhes/exemplos/` — imagens anotadas (verde = gabarito, azul = acerto, vermelho = falso positivo, laranja = não detectado);
- `detalhes/resumo.json` — resumo para preencher o frontend.

In [ ]:
import os
os.environ["YOLO_MODEL_PATH"] = BEST
OUT = f"{DRIVE_DIR}/resultados"
!python -m scripts.evaluate_taco \
    --annotations dataset/subset_3classes/annotations_test.json \
    --images-dir dataset/subset_3classes/test \
    --gamma {GAMMA_TESTE} --conf {CONF} \
    --output {OUT}/taco_metrics.csv \
    --detalhes {OUT}/detalhes --exemplos 12

In [ ]:
import pandas as pd
pd.set_option("display.precision", 3)
df = pd.read_csv(f"{OUT}/taco_metrics.csv")
display(df.pivot(index="class", columns="condition", values=["precision", "recall", "f1_score", "ap50"]))
display(pd.read_csv(f"{OUT}/detalhes/por_iluminacao.csv")
          .pivot(index="faixa", columns="condition", values="f1_score"))

In [ ]:
import glob
from IPython.display import Image, display
for caminho in sorted(glob.glob(f"{OUT}/detalhes/exemplos/*_comparacao.jpg"))[:6]:
    print(os.path.basename(caminho)); display(Image(caminho, width=1200))

## 6.1 As diferenças são reais? (intervalo de confiança)

Reamostra as imagens de teste 2.000 vezes (bootstrap pareado). Se o intervalo inclui 0, a diferença não é significativa com este conjunto de teste.

In [ ]:
import numpy as np, pandas as pd
d = pd.read_csv(f"{OUT}/detalhes/por_imagem.csv")
rng = np.random.default_rng(42)

def f1(s, c):
    tp, fp, fn = s[f"{c}_tp"].sum(), s[f"{c}_fp"].sum(), s[f"{c}_fn"].sum()
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0

amostras = rng.integers(0, len(d), (2000, len(d)))
for c in ("clahe", "gamma"):
    difs = np.array([f1(d.iloc[i], c) - f1(d.iloc[i], "original") for i in amostras])
    lo, hi = np.percentile(difs, [2.5, 97.5])
    print(f"{c} − original: ΔF1 = {f1(d, c) - f1(d, 'original'):+.3f}  "
          f"IC95% [{lo:+.3f}, {hi:+.3f}]  melhor em {(difs > 0).mean():.0%} das reamostragens")

## 7. Próximos passos

- Baixe `best.pt` de `MyDrive/EcoVision/runs/<RUN_NAME>/weights/` e coloque na raiz do projeto local.
  No `.env`: `YOLO_MODEL_PATH=best.pt` — a API e o frontend passam a usar o modelo treinado.
- `resumo.json` e as imagens de `exemplos/` alimentam a seção de resultados do frontend e os slides.